# Testing the eccentric correction of PhenomXHM against PhenomXE

Comparison of `PhenomXHM()` with the eccentric PN phase correction (Yunes et al. 2009, Eq. 4.28, truncated at $\mathcal{O}(e_0^8)$) and `PhenomXE()`, with no BGR correction.

The two models use different phase conventions, so waveforms are never compared point by point: the match is maximized over a constant phase (the `abs` of the overlap) and over a time shift, as in Eq. (4.4) of the XE paper (Ramos-Buades et al.) and Eq. (7.2) of the XHM paper (García-Quirós et al.).

In [ ]:
using GWInference
using LinearAlgebra
using Interpolations # linear_interpolation, used for the noise curve
using Trapz
using Plots
using LaTeXStrings

In [ ]:
# GW170608-like binary, same parameters as in testing_BGR_inclusion.ipynb
m1 = 11.0
m2 = 7.6
mc = (m1*m2)^(3/5) / (m1 + m2)^(1/5)
eta = m1*m2 / (m1 + m2)^2

chi_eff = 0.03
chi1 = chi_eff * (m1 + m2) / m1
chi2 = 0.0

dL = 0.32  # Gpc

e0 = 0.01      # initial eccentricity at the reference frequency f0
f0 = 10.0      # reference GW frequency [Hz]
meanAno = 0.0  # mean anomaly of PhenomXE at fRef (PhenomXHM has no mean anomaly)

# Testing eccentric XE and XHM with no BGR and $e_0 \neq 0$



In [ ]:
# make sure it's the same starting point for both waves:
fmax = 512.0
f_grid = collect(range(f0, fmax, length=2000)) 

hp_XE_0, hc_XE_0 = hphc(PhenomXE(), f_grid, mc, eta, chi1, chi2, dL, 0.0, 0.0, meanAno; fRef=f0)
hp_XHM_0, hc_XHM_0 = hphc(PhenomXHM(), f_grid, mc, eta, chi1, chi2, dL, 0.0; e0=0.0, f0=f0)

fcut = min(_fcut(PhenomXE(), mc, eta, 0.0, 0.0),
               _fcut(PhenomXHM(), mc, eta, 0.0, 0.0)) # must be valid for both in ip integral otherwise it does not make sense so we take the minimum 
    isnothing(fmax) || (fcut = min(fcut, fmax))
print(fcut)

f_grid = f_grid[f_grid .<= fcut];

# getting the noise on the grid:

selected_detector = ETS
psdGrid = linear_interpolation(selected_detector.fNoise, selected_detector.psd, extrapolation_bc = 1.0)(f_grid); # copied from FisherMatrix_Internal

In [ ]:
plot(f_grid, sqrt.(psdGrid),
     title = L"\mathrm{ASD}(f)\ [\mathrm{Hz}^{-1/2}]",
     xlabel = "f (Hz)",
     ylabel = "ASD (ET)",
     lw = 2,
     yscale = :log10,
     legend = false)

In [ ]:
inner_product(a, b) = 4.0 * trapz(f_grid, a .* conj(b) ./ psdGrid) 

In [ ]:
hh_XE_prod = inner_product(hp_XE_0, hp_XE_0)
hh_XHM_prod = inner_product(hp_XHM_0, hp_XHM_0)

In [ ]:
println("rough SNR XE: ", sqrt(real(hh_XE_prod)))
println("rough SNR XHM: ", sqrt(real(hh_XHM_prod)))

In [ ]:
extrema(psdGrid) # variance of psd

In [ ]:
length(hp_XE_0) # checking the length of the array is correctly evaluated

In [ ]:
# if I take the abs value of the inner prod I can cancel out any possible dephasings due to the phase conventions: 

hh_XE_XHM_0_prod = abs(inner_product(hp_XE_0, hp_XHM_0))
faithfulness_0 = hh_XE_XHM_0_prod / sqrt(real(hh_XE_prod) * real(hh_XHM_prod))
println(1-faithfulness_0)

In [ ]:
# now do the test with eccentric models
meanAno = 0.0
hp_XE_e, hc_XE_e = hphc(PhenomXE(), f_grid, mc, eta, chi1, chi2, dL, 0.0, 0.01, meanAno; fRef=f0)
hp_XHM_e, hc_XHM_e = hphc(PhenomXHM(), f_grid, mc, eta, chi1, chi2, dL, 0.0; e0=0.01, f0=f0);

In [ ]:
e0_XHM = range(0.0005, 0.12, step = 1e-4)

norm_XE = real(inner_product(hp_XE_e, hp_XE_e))   # does not change in the loop
faithfulness_i = Float64[]
mismatch_i = Float64[]

for e0_i in e0_XHM
    hp_XHM_e_i, _ = hphc(PhenomXHM(), f_grid, mc, eta, chi1, chi2, dL, 0.0; e0 = e0_i, f0 = f0)
    ip_cross = abs(inner_product(hp_XE_e, hp_XHM_e_i))
    ip_norms = sqrt(norm_XE * real(inner_product(hp_XHM_e_i, hp_XHM_e_i)))
    push!(faithfulness_i, ip_cross / ip_norms)
    push!(mismatch_i, 1 - ip_cross / ip_norms)
end

i_min = argmin(mismatch_i)
println("minimum mismatch ", mismatch_i[i_min], " at e0_XHM = ", e0_XHM[i_min])

In [ ]:
p = plot(e0_XHM, mismatch_i,
     xlabel = L"e_{0,\mathrm{XHM}}",
     ylabel = L"1 - \mathrm{faithfulness}",
     xscale = :log10,
     yscale = :log10,
     lw = 3,
     color = :royalblue,
     label = L"\mathrm{XHM}\ \mathrm{vs}\ \mathrm{XE}",
     legend = :topleft,
     size = (1000, 700),
     guidefontsize = 16,
     tickfontsize = 13,
     legendfontsize = 12,
     left_margin = 5Plots.mm,
     bottom_margin = 5Plots.mm,
     framestyle = :box,
     grid = :both,
     minorgrid = true)

vline!(p, [0.01],
     color = :darkorange,
     ls = :dash,
     lw = 2,
     label = L"e_{\mathrm{XE}} = 0.01")

scatter!(p, [e0_XHM[i_min]], [mismatch_i[i_min]],
     color = :crimson,
     markersize = 8,
     markershape = :star5,
     markerstrokecolor = :black,
     label = L"\mathrm{minimum \ mismatch}")

hline!(p, [5.6e-4],
     color = :darkgreen,
     ls = :dot,
     lw = 2.5,
     label = L"\mathrm{Circular\ baseline}\ (e_\mathrm{XE, XHM} =0)")

display(p)

### Scan at different $e_\mathrm{XE}$

Same scan as above, written as a function of $e_\mathrm{XE}$ and with the x-axis normalized: $e_{0,\mathrm{XHM}}/e_\mathrm{XE}$. If $e_0$ and $e$ are the same quantity, the minimum sits at a ratio of 1 for every $e_\mathrm{XE}$.

In [ ]:
function scan_e0(e_XE, ratios; meanAno = 0.0)
    # XE is evaluated once at e_XE, XHM is re-evaluated for every e0 = ratio * e_XE
    hp_XE_e, _ = hphc(PhenomXE(), f_grid, mc, eta, chi1, chi2, dL, 0.0, e_XE, meanAno; fRef = f0)
    norm_XE = real(inner_product(hp_XE_e, hp_XE_e))   # does not change in the loop
    mismatch = Float64[]
    for r in ratios
        hp_XHM_i, _ = hphc(PhenomXHM(), f_grid, mc, eta, chi1, chi2, dL, 0.0; e0 = r * e_XE, f0 = f0)
        ip_cross = abs(inner_product(hp_XE_e, hp_XHM_i))
        ip_norms = sqrt(norm_XE * real(inner_product(hp_XHM_i, hp_XHM_i)))
        push!(mismatch, 1 - ip_cross / ip_norms)
    end
    return mismatch
end

In [ ]:
ratios = range(0.05, 3.0, step = 0.005)
e_XE_list = [0.01, 0.02, 0.05]

mismatch_scan = Dict{Float64, Vector{Float64}}()
for e_XE in e_XE_list
    mismatch_scan[e_XE] = scan_e0(e_XE, ratios)
    i = argmin(mismatch_scan[e_XE])
    println("e_XE = $e_XE: min mismatch = ", mismatch_scan[e_XE][i], " at e0_XHM/e_XE = ", ratios[i])
end

In [ ]:
p = plot(xlabel = L"e_{0,\mathrm{XHM}} / e_{\mathrm{XE}}",
         ylabel = L"1 - \mathrm{faithfulness}",
         xscale = :log10, yscale = :log10,
         legend = :topleft, size = (1000, 700),
         guidefontsize = 16, tickfontsize = 13, legendfontsize = 12,
         left_margin = 5Plots.mm, bottom_margin = 5Plots.mm,
         framestyle = :box, grid = :both, minorgrid = true)

for e_XE in e_XE_list
    m = mismatch_scan[e_XE]
    plot!(p, ratios, m, lw = 3, label = L"e_\mathrm{XE} = %$e_XE")
    scatter!(p, [ratios[argmin(m)]], [minimum(m)], markershape = :star5, markersize = 8,
             markerstrokecolor = :black, label = "")
end

vline!(p, [1.0], color = :black, ls = :dash, lw = 2, label = L"e_0 = e_\mathrm{XE}")
hline!(p, [5.6e-4], color = :darkgreen, ls = :dot, lw = 2.5, label = "circular baseline")
display(p)

### Scan with a time shift

Same scan at $e_\mathrm{XE} = 0.01$, but the match is now maximized over a time shift $\tau$ as well as the constant phase (the `abs`), as in Eq. (4.4) of the XE paper and Eq. (7.2) of the XHM paper. The shift is applied to the XHM waveform as $e^{2\pi i f \tau}$.

In [ ]:
τ_s = range(-8e-4, 8e-4, step = 1e-6)   # trial time shifts [s]: the baseline best value is about -6.5e-5 s

function match_tau(a, b, τ_s)

    # a time shift does not change the norm of b, so the norms are computed once
    ip_norms = sqrt(real(inner_product(a, a)) * real(inner_product(b, b)))

    mismatch_i = Float64[]

    for τ in τ_s
        b_τ = b .* exp.(2π * im .* f_grid .* τ) # apply shift to b waveform, make sure it's XHM -> convention with +
        faithfulness_i = abs(inner_product(a, b_τ)) / ip_norms
        push!(mismatch_i, 1 - faithfulness_i)
    end

    i = argmin(mismatch_i)
    return mismatch_i[i], τ_s[i]      # best mismatch and the shift where it occurs
end

In [ ]:
# tests of match_tau

# 1) known phase and shift tau = -3e-4: expected mismatch ~ 0
a_test = hp_XE_0
b_test = a_test .* exp(1im * 0.7) .* exp.(2π * im .* f_grid .* 3e-4)
println("known shift: ", match_tau(a_test, b_test, range(-1e-3, 1e-3, step = 1e-5)))

# 2) circular baseline
println("circular baseline: ", match_tau(hp_XE_0, hp_XHM_0, range(-1e-3, 1e-3, step = 5e-6)))

In [ ]:
function scan_e0_tau(e_XE, ratios, τ_s; meanAno = 0.0)
    # XE is evaluated once at e_XE, XHM is re-evaluated for every e0 = ratio * e_XE
    hp_XE_e, _ = hphc(PhenomXE(), f_grid, mc, eta, chi1, chi2, dL, 0.0, e_XE, meanAno; fRef = f0) # find the waveform -> 
    mismatch = Float64[] # declare mismatch and best time shift
    best_τ = Float64[]

    for r in ratios
        hp_XHM_i, _ = hphc(PhenomXHM(), f_grid, mc, eta, chi1, chi2, dL, 0.0; e0 = r * e_XE, f0 = f0) # again evaluate the XHM e-waveform
        m, τ = match_tau(hp_XE_e, hp_XHM_i, τ_s) # return the matching tau shift for each of the eccentricity vals
        push!(mismatch, m)
        push!(best_τ, τ)
    end
    return mismatch, best_τ # return mismatch value and best tau found for different e_XE / e_XHM
end

ratios_τ = range(0.2, 3.0, step = 0.01) # still ratios of e_XE / e_XHM
mismatch_τ, best_τ = scan_e0_tau(0.01, ratios_τ, τ_s)

i = argmin(mismatch_τ)
println("with time shift τ: min mismatch = ", mismatch_τ[i], " at e0_XHM/e_XE = ", ratios_τ[i])
println("at ratio 1: ", mismatch_τ[argmin(abs.(ratios_τ .- 1.0))])
# if the best tau sits on an edge of τ_s, the range is too narrow --> Claude :)
println("best tau between ", minimum(best_τ), " and ", maximum(best_τ), "   (scan edges: ", first(τ_s), ", ", last(τ_s), ")")

In [ ]:
p = plot(xlabel = L"e_{0,\mathrm{XHM}} / e_{\mathrm{XE}}",
         ylabel = L"1 - \mathrm{faithfulness}",
         yscale = :log10,
         legend = :bottomright, size = (1000, 700),
         guidefontsize = 16, tickfontsize = 13, legendfontsize = 12,
         left_margin = 5Plots.mm, bottom_margin = 5Plots.mm,
         framestyle = :box, grid = :both, minorgrid = true)

# abs only (previous scan) and abs + time shift
plot!(p, ratios, mismatch_scan[0.01], lw = 3, label = L"\mathrm{phase\ match\ only}")
plot!(p, ratios_τ, mismatch_τ, lw = 3, label = L"\mathrm{phase\ match\ + time\ shift}")
scatter!(p, [ratios_τ[argmin(mismatch_τ)]], [minimum(mismatch_τ)], markershape = :star5, markersize = 8,
         markerstrokecolor = :black, label = "")

vline!(p, [1.0], color = :black, ls = :dash, lw = 2, label = L"e_0 = e")
#plot!(p, xlims = (0.7, 1.4))
display(p)

### Scan with a time shift at $e_\mathrm{XE} = 0.02$ and $0.05$

Same scan as at $e_\mathrm{XE} = 0.01$ (same `τ_s` and `ratios_τ`), repeated for the two larger eccentricities. If $e_0$ and $e$ are the same quantity the minimum should sit at a ratio of 1 for every $e_\mathrm{XE}$. At larger $e$ the dephasing grows as $e^2$, so the valley is narrower and the minimum is better determined.

In [ ]:
e_XE_list_τ = [0.02, 0.05]

# the 0.01 result is already computed above
mismatch_τ_scan = Dict{Float64, Vector{Float64}}(0.01 => mismatch_τ)
best_τ_scan = Dict{Float64, Vector{Float64}}(0.01 => best_τ)

for e_XE in e_XE_list_τ
    mismatch_τ_scan[e_XE], best_τ_scan[e_XE] = scan_e0_tau(e_XE, ratios_τ, τ_s)

    i = argmin(mismatch_τ_scan[e_XE])
    println("e_XE = $e_XE, with time shift τ: min mismatch = ", mismatch_τ_scan[e_XE][i], " at e0_XHM/e_XE = ", ratios_τ[i])
    println("   at ratio 1: ", mismatch_τ_scan[e_XE][argmin(abs.(ratios_τ .- 1.0))])
    # if the best tau sits on an edge of τ_s, the range is too narrow
    println("   best tau between ", minimum(best_τ_scan[e_XE]), " and ", maximum(best_τ_scan[e_XE]), "   (scan edges: ", first(τ_s), ", ", last(τ_s), ")")
end

In [ ]:
# one plot per e_XE, same style as above: phase match only (previous scan) vs phase match + time shift
for e_XE in e_XE_list_τ
    p = plot(xlabel = L"e_{0,\mathrm{XHM}} / e_{\mathrm{XE}}",
             ylabel = L"1 - \mathrm{faithfulness}",
             title = L"e_\mathrm{XE} = %$e_XE",
             yscale = :log10,
             legend = :bottomright, size = (1000, 700),
             guidefontsize = 16, tickfontsize = 13, legendfontsize = 12,
             left_margin = 5Plots.mm, bottom_margin = 5Plots.mm,
             framestyle = :box, grid = :both, minorgrid = true)

    plot!(p, ratios, mismatch_scan[e_XE], lw = 3, label = L"\mathrm{phase\ match\ only}")
    plot!(p, ratios_τ, mismatch_τ_scan[e_XE], lw = 3, label = L"\mathrm{phase\ match\ + time\ shift}")
    scatter!(p, [ratios_τ[argmin(mismatch_τ_scan[e_XE])]], [minimum(mismatch_τ_scan[e_XE])], markershape = :star5, markersize = 8,
             markerstrokecolor = :black, label = "")

    vline!(p, [1.0], color = :black, ls = :dash, lw = 2, label = L"e_0 = e")
    display(p)
end

In [ ]:
# the three eccentricities together, phase match + time shift
p = plot(xlabel = L"e_{0,\mathrm{XHM}} / e_{\mathrm{XE}}",
         ylabel = L"1 - \mathrm{faithfulness}",
         yscale = :log10,
         legend = :bottomright, size = (1000, 700),
         guidefontsize = 16, tickfontsize = 13, legendfontsize = 12,
         left_margin = 5Plots.mm, bottom_margin = 5Plots.mm,
         framestyle = :box, grid = :both, minorgrid = true)

for e_XE in sort(collect(keys(mismatch_τ_scan)))
    m = mismatch_τ_scan[e_XE]
    plot!(p, ratios_τ, m, lw = 3, label = L"e_\mathrm{XE} = %$e_XE")
    scatter!(p, [ratios_τ[argmin(m)]], [minimum(m)], markershape = :star5, markersize = 8,
             markerstrokecolor = :black, label = "")
end

vline!(p, [1.0], color = :black, ls = :dash, lw = 2, label = L"e_0 = e")
display(p)